### VaR Model test

In [1]:
import numpy as np
import pandas as pd

from tvDatafeed import TvDatafeed, Interval

tv = TvDatafeed()

kospi = tv.get_hist(
    symbol = 'KOSPI',
    exchange = 'TVC',
    interval = Interval.in_daily,
    n_bars = 13000
)
kospi.index = pd.to_datetime(
    kospi.index.strftime('%Y-%m-%d'),
)

In [4]:
from src.risk.value_at_risk import StockVaR

value_at_risk_model = StockVaR(
    prices = kospi['close'],
    position_value = 100000000
)

### Historical VaR

In [7]:
value_at_risk_model.historical_var(
    holding_period = 20
) # historical VaR, holding periods = 20 business days

10211635.13199176

### Monte Carlo Simulation VaR

In [10]:
value_at_risk_model.monte_carlo_var(
    holding_period = 20
) # monte carlo VaR, holding periods = 20 business days

10392306.41449174

change the stochastic engine

In [12]:
from src.monte_carlo import *

# student t distribution
student_engine = StudentTMonteCarlo(
    n_simulations = 10000,
    random_seed = 42
)

value_at_risk_model.monte_carlo_var(
    holding_period = 20,
    mc_engine = student_engine
)

9899777.676093074

In [13]:
# Sequential Bootstrapping

bootstrap_engine = BootstrapMonteCarlo(
    n_simulations = 10000,
    random_seed = 42
)

value_at_risk_model.monte_carlo_var(
    holding_period = 20,
    mc_engine = bootstrap_engine
)

10202248.391551118

In [16]:
# Kernel Density
kde_engine = KernelDensityMonteCarlo(
    n_simulations = 10000,
    random_seed = 42
)

value_at_risk_model.monte_carlo_var(
    holding_period = 20,
    mc_engine = kde_engine,
    bandwidth = 0.2
)

10652616.692784596

### Parametic VaR

In [17]:
value_at_risk_model.parametric_var(
    holding_period = 20
)

10318528.654852867

### Stress VaR

In [19]:
value_at_risk_model.stressed_var(
    holding_period = 20,
    stress_returns = kospi['close'].pct_change().loc[
        '2020-03':'2020-04'
    ]
) # stress VaR, during COVID-19, 20 business days

21574044.40469737

In [24]:
value_at_risk_model.stressed_var(
    holding_period = 20,
    stress_returns = kospi['close'].pct_change().loc[
        '2025-03':'2025-04'
    ]
) # stress VaR, during tariff shock, 20 business days

7647421.991546902

### Expected Shortfall (cVaR)

In [28]:
value_at_risk_model.expected_shortfall(
    holding_period = 20
)

16081916.749653457

### VaR Summary

In [32]:
value_at_risk_model.summary(holding_period = 20)

{'historical_var': 10211635.13199176,
 'parametric_var': 10318528.654852867,
 'monte_carlo_var': 10343901.843544973,
 'historical_es': 16081916.749653457,
 'parametric_es': 13177187.496032622,
 'monte_carlo_es': 13101464.813951494}